# Backtests vs Live Trading Performance Analysis

This notebook compares the performance of a trading strategy in two settings:

1. **Backtest performance**: how the strategy performed on historical data
2. **Live trading performance**: how the same strategy performed after execution in real market conditions

The purpose is to show why a strategy that looks strong in a backtest may perform differently in live trading. Backtests can miss real-world frictions such as slippage, transaction costs, latency, changing market conditions, and execution errors.

The notebook includes:

1. Simulated backtest and live trade results
2. Return comparison
3. Equity curve comparison
4. Drawdown analysis
5. Sharpe ratio and Sortino ratio
6. Win rate, profit factor, and average trade return
7. Slippage and execution gap analysis
8. Main interpretation and limitations

This is designed as a portfolio-ready trading analytics project.

## 1. Import Libraries

We begin by importing the main Python libraries used for data analysis and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option("display.max_columns", 50)

## 2. Create Sample Backtest and Live Trading Data

In a real project, this section would load two files:

- A backtest trade log
- A live trading execution log

For this notebook, we create realistic simulated data so the analysis can run without external files. The live results are intentionally slightly worse because live trading usually faces slippage, fees, latency, missed fills, and changing market conditions.

In [ ]:
n_trades = 350

dates = pd.date_range(start="2025-01-01", periods=n_trades, freq="D")

# Backtest returns: slightly optimistic
backtest_returns = np.random.normal(loc=0.0032, scale=0.018, size=n_trades)

# Add a few strong winning periods and losing periods
backtest_returns[40:55] += 0.015
backtest_returns[160:175] -= 0.020
backtest_returns[250:265] += 0.010

# Live returns: same general strategy, but reduced by execution costs and slippage
slippage = np.random.normal(loc=0.0010, scale=0.0015, size=n_trades)
execution_noise = np.random.normal(loc=0.0, scale=0.006, size=n_trades)
live_returns = backtest_returns - slippage + execution_noise

# Some trades are missed or poorly executed in live trading
missed_trade_idx = np.random.choice(np.arange(n_trades), size=25, replace=False)
live_returns[missed_trade_idx] = np.random.normal(loc=-0.001, scale=0.006, size=len(missed_trade_idx))

trades = pd.DataFrame({
    "date": dates,
    "backtest_return": backtest_returns,
    "live_return": live_returns,
    "estimated_slippage": slippage,
})

trades["performance_gap"] = trades["backtest_return"] - trades["live_return"]

trades.head()

## 3. Create Equity Curves

An equity curve shows how capital grows or declines over time. It is one of the most useful ways to compare backtest and live performance visually.

In [ ]:
initial_capital = 10000

trades["backtest_equity"] = initial_capital * (1 + trades["backtest_return"]).cumprod()
trades["live_equity"] = initial_capital * (1 + trades["live_return"]).cumprod()

trades[["date", "backtest_equity", "live_equity"]].head()

## 4. Plot Backtest vs Live Equity Curves

If the live equity curve is much lower than the backtest equity curve, the difference may suggest overfitting, unrealistic assumptions, execution costs, or regime change.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(trades["date"], trades["backtest_equity"], label="Backtest Equity")
plt.plot(trades["date"], trades["live_equity"], label="Live Equity")
plt.title("Backtest vs Live Trading Equity Curves")
plt.xlabel("Date")
plt.ylabel("Account Value")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Define Performance Metrics

We define helper functions for common trading performance metrics:

- Total return
- Annualized return
- Annualized volatility
- Sharpe ratio
- Sortino ratio
- Maximum drawdown
- Win rate
- Average trade return
- Profit factor

These metrics give a more complete view than total return alone.

In [ ]:
def max_drawdown(return_series):
    equity = (1 + return_series).cumprod()
    running_max = equity.cummax()
    drawdown = equity / running_max - 1
    return drawdown.min()


def sharpe_ratio(return_series, periods_per_year=252):
    if return_series.std() == 0:
        return np.nan
    return (return_series.mean() / return_series.std()) * np.sqrt(periods_per_year)


def sortino_ratio(return_series, periods_per_year=252):
    downside = return_series[return_series < 0]
    if downside.std() == 0:
        return np.nan
    return (return_series.mean() / downside.std()) * np.sqrt(periods_per_year)


def profit_factor(return_series):
    gains = return_series[return_series > 0].sum()
    losses = abs(return_series[return_series < 0].sum())
    if losses == 0:
        return np.nan
    return gains / losses


def performance_summary(return_series, label):
    total_return = (1 + return_series).prod() - 1
    annualized_return = (1 + total_return) ** (252 / len(return_series)) - 1
    annualized_volatility = return_series.std() * np.sqrt(252)
    return {
        "Strategy Version": label,
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Sharpe Ratio": sharpe_ratio(return_series),
        "Sortino Ratio": sortino_ratio(return_series),
        "Max Drawdown": max_drawdown(return_series),
        "Win Rate": (return_series > 0).mean(),
        "Average Trade Return": return_series.mean(),
        "Profit Factor": profit_factor(return_series)
    }

## 6. Compare Performance Metrics

The performance table shows whether the live version preserves the same quality as the backtest.

In [ ]:
summary = pd.DataFrame([
    performance_summary(trades["backtest_return"], "Backtest"),
    performance_summary(trades["live_return"], "Live Trading")
])

summary_rounded = summary.copy()
for col in summary_rounded.columns:
    if col != "Strategy Version":
        summary_rounded[col] = summary_rounded[col].round(4)

summary_rounded

## 7. Drawdown Analysis

Drawdown measures how far the strategy falls from its previous peak. A strategy may have a high return but still be hard to trade if the drawdowns are large.

In [ ]:
def drawdown_series(return_series):
    equity = (1 + return_series).cumprod()
    running_max = equity.cummax()
    return equity / running_max - 1

trades["backtest_drawdown"] = drawdown_series(trades["backtest_return"])
trades["live_drawdown"] = drawdown_series(trades["live_return"])

plt.figure(figsize=(10, 5))
plt.plot(trades["date"], trades["backtest_drawdown"], label="Backtest Drawdown")
plt.plot(trades["date"], trades["live_drawdown"], label="Live Drawdown")
plt.title("Backtest vs Live Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Return Distribution Comparison

A return distribution helps show whether live trading has more negative trades, fewer large winners, or higher volatility than the backtest.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(trades["backtest_return"], bins=35, alpha=0.7, label="Backtest")
plt.hist(trades["live_return"], bins=35, alpha=0.7, label="Live Trading")
plt.title("Return Distribution: Backtest vs Live")
plt.xlabel("Trade Return")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Performance Gap Analysis

The performance gap is defined as:

`backtest return - live return`

A positive gap means the backtest performed better than live trading for that trade. Large positive gaps may indicate slippage, missed fills, delayed execution, or unrealistic backtest assumptions.

In [ ]:
gap_summary = trades["performance_gap"].describe().to_frame(name="Performance Gap")
gap_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(trades["date"], trades["performance_gap"])
plt.axhline(0, linestyle="--")
plt.title("Trade-by-Trade Performance Gap")
plt.xlabel("Date")
plt.ylabel("Backtest Return - Live Return")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Slippage Analysis

Slippage is the difference between the expected execution price and the actual execution price. In live trading, slippage can quietly destroy a strategy that looks profitable in a backtest.

In [ ]:
slippage_stats = trades["estimated_slippage"].describe().to_frame(name="Estimated Slippage")
slippage_stats

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(trades["estimated_slippage"], trades["performance_gap"], alpha=0.7)
plt.title("Slippage vs Performance Gap")
plt.xlabel("Estimated Slippage")
plt.ylabel("Performance Gap")
plt.tight_layout()
plt.show()

print("Correlation between slippage and performance gap:", round(trades["estimated_slippage"].corr(trades["performance_gap"]), 4))

## 11. Monthly Performance Comparison

Monthly aggregation helps identify whether the performance gap is stable or concentrated in certain periods.

In [ ]:
monthly = trades.copy()
monthly["month"] = monthly["date"].dt.to_period("M").astype(str)

monthly_summary = monthly.groupby("month").agg(
    backtest_monthly_return=("backtest_return", lambda x: (1 + x).prod() - 1),
    live_monthly_return=("live_return", lambda x: (1 + x).prod() - 1),
    average_gap=("performance_gap", "mean"),
    trades=("date", "count")
).reset_index()

monthly_summary

In [ ]:
plt.figure(figsize=(10, 5))
x = np.arange(len(monthly_summary))
width = 0.35

plt.bar(x - width / 2, monthly_summary["backtest_monthly_return"], width, label="Backtest")
plt.bar(x + width / 2, monthly_summary["live_monthly_return"], width, label="Live Trading")
plt.title("Monthly Return Comparison")
plt.xlabel("Month")
plt.ylabel("Monthly Return")
plt.xticks(x, monthly_summary["month"], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 12. Diagnose Possible Causes of Live Underperformance

A live strategy can underperform its backtest for several reasons. This section creates a simple diagnostic table that connects symptoms to possible explanations.

In [ ]:
diagnostics = pd.DataFrame({
    "Observation": [
        "Live return is lower than backtest return",
        "Live drawdown is larger than backtest drawdown",
        "Performance gap is positive on average",
        "Some live trades are much worse than expected",
        "Monthly gap varies over time"
    ],
    "Possible Explanation": [
        "Fees, slippage, spread costs, or optimistic backtest assumptions",
        "Risk model may underestimate live volatility or changing market conditions",
        "Execution costs may not be fully included in the backtest",
        "Latency, missed fills, bad fills, or exchange/order-book effects",
        "Market regime changes or instability in the strategy edge"
    ],
    "Suggested Check": [
        "Add realistic trading costs and rerun the backtest",
        "Compare live and historical volatility during drawdown periods",
        "Measure trade-level slippage and commissions",
        "Audit execution logs and order timestamps",
        "Run rolling-window performance analysis"
    ]
})

diagnostics

## 13. Save Analysis Outputs

For a portfolio or report, it is useful to save the main tables as CSV files.

In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

trades.to_csv(output_dir / "backtest_vs_live_trade_results.csv", index=False)
summary_rounded.to_csv(output_dir / "backtest_vs_live_performance_summary.csv", index=False)
monthly_summary.to_csv(output_dir / "backtest_vs_live_monthly_summary.csv", index=False)

print("Output files saved in the output folder.")

## 14. Main Conclusion

This notebook shows how to compare backtest performance with live trading performance in a structured way. The analysis focuses on both return and risk, including equity curves, drawdowns, Sharpe ratio, Sortino ratio, win rate, profit factor, and trade-by-trade performance gaps.

The key lesson is that a strong backtest does not automatically mean strong live performance. Backtests often assume cleaner execution than real markets allow. Live trading introduces slippage, fees, latency, missed fills, emotional interference, exchange constraints, and changing market conditions.

A strategy should therefore be judged not only by historical results, but also by how well it survives live execution. A good research process should continuously compare backtest expectations with live results and update assumptions when the gap becomes too large.